In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from datetime import datetime, timedelta

In [0]:
# ── Configuration ────────────────────────────────────────────────────────
kafka_bootstrap_servers = "pkc-619z3.us-east1.gcp.confluent.cloud:9092"
kafka_topic            = "xml_topic"
xml_file_path          = "/Volumes/streaming/landing_vol/kafka/xml/"     
row_tag                = "order"   
# ── Confluent Cloud credentials (SASL_SSL) ───────────────────────────────
#   Replace these with your Confluent Cloud API key & secret, or use dbutils.secrets.get()
confluent_api_key    = "ZATXMTIJZNGATCSM"
confluent_api_secret = "cfltBHf2fbnYXbaxEVai4Ia2L3P1nczO1ATF1PFs16OoVGqo/qeaIdsAXBkoKqdA"

# -------------------------------------------------------------------------
# 4: Load XML and Publish to Event Hub
# -------------------------------------------------------------------------
xml_raw=spark.read.text(xml_file_path,wholetext=True)
xml_records = (
    xml_raw
    .select(
        regexp_extract_all(col("value"), lit(r"(<order>[\s\S]*?</order>)"), 1).alias("records")
    )
    .select(explode(col("records")).alias("value"))
    .filter(col("value") != "")
)

# Prepare for Event Hub: key = file identifier, value = individual XML record
kafka_df = (
    xml_records
    .withColumn("key", lit("sales"))
    .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
)

print(f"XML file loaded from: {xml_file_path}")
print(f"Records to send: {kafka_df.count()}")
print("Message preview (first 200 chars):")
first_row = kafka_df.first()
if first_row:
    print(first_row["value"][:200] + "...")
# ── Convert each parsed row to a JSON string for the Kafka payload ────────
#   Kafka expects a column named "value" (binary) for the message body.



# ── Write to Kafka (batch) ───────────────────────────────────────────────
(
    kafka_df.write
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("topic", kafka_topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{confluent_api_key}" password="{confluent_api_secret}";')
    .save()
)

print(f"Wrote  records from XML '{xml_file_path}' to Kafka topic "
      f"'{kafka_topic}' on cluster '{kafka_bootstrap_servers}'.")